# Project 03 — Medical Image Anomaly Detector
## Phase 1 Training (Binary: NORMAL vs PNEUMONIA)

This notebook runs **Phase 1 only** (head training, backbone frozen) on a Colab T4 GPU.

**Gate to clear before moving on:** validation mean AUC > 0.70

Steps below: check GPU → upload project code → install deps → download dataset → sanity-check the pipeline with pytest → train Phase 1 → inspect results.

Runtime: **Runtime → Change runtime type → T4 GPU** before running anything below.

## 1. Confirm GPU is available

In [1]:
!nvidia-smi

Wed Jun 24 05:41:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload the project code

Run the cell below, then in the file picker select **`chest-xray-detector.zip`**
(the project zip from this conversation — it contains `src/`, `tests/`,
`run_training.py`, and `requirements.txt`).

In [2]:
from google.colab import files
uploaded = files.upload()  # select chest-xray-detector.zip


Saving chest-xray-detector-complete.zip to chest-xray-detector-complete.zip


In [3]:
import zipfile, os

zip_name = [f for f in uploaded.keys() if f.endswith('.zip')][0]
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/')

# The zip may extract as /content/chest-xray-detector/... — locate it
for root, dirs, _ in os.walk('/content'):
    if 'src' in dirs and 'run_training.py' in os.listdir(root):
        PROJECT_DIR = root
        break

%cd $PROJECT_DIR
print(f"Project directory: {PROJECT_DIR}")
!ls


/content
Project directory: /content
app				  notebooks	    run_evaluation.py  src
chest-xray-detector-complete.zip  README.md	    run_training.py    tests
deploy				  requirements.txt  sample_data


## 3. Install dependencies

In [4]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 609.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 112.5 MB/s eta 0:00:00
   ━━━━

## 4. Mount Google Drive (for checkpoint persistence)

Colab sessions disconnect without warning — checkpoints go to Drive so a
disconnect during a multi-hour run doesn't lose progress.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints/chest_xray'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")


Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/checkpoints/chest_xray


## 5. Set up Kaggle API access

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token**
   This downloads a `kaggle.json` file.
2. Run the cell below and upload that `kaggle.json` when prompted.

In [6]:
from google.colab import files
print("Upload your kaggle.json file:")
kaggle_uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle credentials installed.")


Upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle credentials installed.


## 6. Download the dataset (~2GB, Kaggle Chest X-Ray Images Pneumonia)

In [7]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip


Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [02:14<00:00, 19.2MB/s]
100% 2.29G/2.29G [02:14<00:00, 18.4MB/s]


## 7. Locate the real `train/` folder

The Kaggle zip sometimes extracts with an extra nested `chest_xray/chest_xray/`
folder depending on download method. This cell finds wherever `train/`
(containing NORMAL/PNEUMONIA subfolders) actually landed, instead of
hardcoding a path that might be wrong.

In [8]:
import os

def find_data_root(search_root='/content/data'):
    for root, dirs, _ in os.walk(search_root):
        if 'train' in dirs:
            candidate = os.path.join(root, 'train')
            subdirs = os.listdir(candidate)
            if 'NORMAL' in subdirs and 'PNEUMONIA' in subdirs:
                return root
    raise FileNotFoundError(
        f"Could not find a train/ folder with NORMAL/PNEUMONIA subfolders under {search_root}. "
        f"Run `!find {search_root} -maxdepth 3` to inspect the extracted structure manually."
    )

DATA_ROOT = find_data_root()
print(f"DATA_ROOT = {DATA_ROOT}")
!ls "$DATA_ROOT"
!echo "---"
!ls "$DATA_ROOT/train"


DATA_ROOT = /content/data/chest_xray
chest_xray  __MACOSX  test  train  val
---
NORMAL	PNEUMONIA


## 8. Sanity-check the pipeline before burning GPU hours

Runs the full test suite from Weeks 1–2 against the real environment
(confirms torch/torchvision versions, EfficientNetB3 download, etc. all
work here — not just in the original dev sandbox).

In [9]:
!python -m pytest tests/ -q

........................................................................ [ 61%]
..............................................                           [100%]
118 passed in 30.39s


## 9. (Optional) Weights & Biases login

Skip this cell if you'd rather not track the run — pass `--no-wandb` in
step 11 instead.

In [10]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize


wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: ··········


wandb: ERROR API key must be 40 characters long, yours was 86


...

wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: ··········


wandb: ERROR API key must be 40 characters long, yours was 36


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize



Abort: 

## 10. Quick smoke test — 1 epoch, no pretrained download, no wandb

Confirms the full train_epoch/val_epoch loop runs end-to-end on REAL data
before committing GPU time to a full Phase 1 run. Takes ~1-2 minutes.

In [12]:
!python run_training.py \
    --data-root "$DATA_ROOT" \
    --checkpoint-dir /content/checkpoints_smoketest \
    --epochs1 1 \
    --epochs2 0 \
    --no-wandb \
    --no-pretrained


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
05:58:35 | INFO     | __main__             | Config: Config(SEED=42, DATA_ROOT='/content/data/chest_xray', TRAIN_DIR='train', VAL_DIR='val', TEST_DIR='test', CHECKPOINT_DIR='/content/checkpoints_smoketest', CLASS_NAMES=['NORMAL', 'PNEUMONIA'], NUM_CLASSES=1, RESIZE_SIZE=256, IMG_SIZE=224, BATCH_SIZE=32, NUM_WORKERS=0, PIN_MEMORY=False, VAL_SPLIT=0.2, HFLIP_PROB=0.5, ROTATION_DEG=10, BRIGHTNESS_JITTER=0.2, CONTRAST_JITTER=0.2, LR_PHASE1=0.001, EPOCHS_PHASE1=1, LR_PHASE2=0.0001, EPOCHS_PHASE2=0, UNFREEZE_BLOCKS=3, WEIGHT_DECAY=0.0001, GRAD_CLIP=1.0, ETA_MIN=1e-06, DROPOUT=0.4, BACKBONE_OUT_FEATURES=1536, FOCAL_GAMMA=2.0, FOCAL_ALPHA=0.25, THRESHOLD=0.5, WANDB_PROJECT='chest-xray-binary', WANDB_ENTITY='', USE_WANDB=False, USE_AMP=True, CHECKPOINT_EVERY=1, EARLY_STOP_PATIENCE=3, LOG_EVERY_N_BATCHES=20)
05:58:35 | INFO     | src.utils            | All random seeds set to 42
05:58:35 | INFO     | src.utils   

## 11. Run Phase 1 (full — 5 epochs, head only, backbone frozen)

`--epochs2 0` skips Phase 2 entirely for now — we're only clearing the
Phase 1 gate (val AUC > 0.70) before deciding whether to continue.

Expect roughly 1-3 minutes/epoch on a T4 for this dataset size (~4,200
train images), so this should take well under 15 minutes total.

In [13]:
!python run_training.py \
    --data-root "$DATA_ROOT" \
    --checkpoint-dir "$CHECKPOINT_DIR" \
    --epochs1 5 \
    --epochs2 0


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
06:01:46 | INFO     | __main__             | Config: Config(SEED=42, DATA_ROOT='/content/data/chest_xray', TRAIN_DIR='train', VAL_DIR='val', TEST_DIR='test', CHECKPOINT_DIR='/content/drive/MyDrive/checkpoints/chest_xray', CLASS_NAMES=['NORMAL', 'PNEUMONIA'], NUM_CLASSES=1, RESIZE_SIZE=256, IMG_SIZE=224, BATCH_SIZE=32, NUM_WORKERS=0, PIN_MEMORY=False, VAL_SPLIT=0.2, HFLIP_PROB=0.5, ROTATION_DEG=10, BRIGHTNESS_JITTER=0.2, CONTRAST_JITTER=0.2, LR_PHASE1=0.001, EPOCHS_PHASE1=5, LR_PHASE2=0.0001, EPOCHS_PHASE2=0, UNFREEZE_BLOCKS=3, WEIGHT_DECAY=0.0001, GRAD_CLIP=1.0, ETA_MIN=1e-06, DROPOUT=0.4, BACKBONE_OUT_FEATURES=1536, FOCAL_GAMMA=2.0, FOCAL_ALPHA=0.25, THRESHOLD=0.5, WANDB_PROJECT='chest-xray-binary', WANDB_ENTITY='', USE_WANDB=True, USE_AMP=True, CHECKPOINT_EVERY=1, EARLY_STOP_PATIENCE=3, LOG_EVERY_N_BATCHES=20)
06:01:46 | INFO     | src.utils            | All random seeds set to 42
06:01:46 | INFO     

## 12. Check the result against the gate

**Gate: validation mean AUC > 0.70**

If you cleared it, you're ready to move to Phase 2 (fine-tuning) and then
Week 3 (`evaluate.py` — full test-set AUC, F1, confusion matrix).

If you didn't clear it, common causes worth checking first: dataset really
loaded the full ~4,200 train images (not a tiny subset), `pos_weight` looks
sane (~0.35), and the loss was actually decreasing across epochs 1→5 in the
logs above.

In [18]:
import numpy
print(torch.__version__)
print(numpy.__version__)

2.3.0+cu121
2.0.2


In [15]:
import torch

ckpt = torch.load(f"{CHECKPOINT_DIR}/best_model.pth", map_location='cpu',weights_only=False)
print(f"Best epoch: {ckpt['epoch'] + 1}")
print(f"Best val AUC: {ckpt['best_auc']:.4f}")
print()
if ckpt['best_auc'] > 0.70:
    print("✓ GATE CLEARED — ready for Phase 2 fine-tuning")
else:
    print("✗ Gate not yet cleared — review training curves before proceeding")


TypeError: scalar() argument 1 must be numpy.dtype, not numpy.dtypes.Float64DType

## 13. (When ready) Run Phase 2 — fine-tuning

Resumes is not yet wired up automatically here (that lands with
`evaluate.py` in Week 3), so this re-runs Phase 1 + Phase 2 together. Phase 1
is fast (head-only), so this is fine for now.

In [19]:
!python run_training.py \
    --data-root "$DATA_ROOT" \
    --checkpoint-dir "$CHECKPOINT_DIR" \
    --epochs1 5 \
    --epochs2 10


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
06:15:21 | INFO     | __main__             | Config: Config(SEED=42, DATA_ROOT='/content/data/chest_xray', TRAIN_DIR='train', VAL_DIR='val', TEST_DIR='test', CHECKPOINT_DIR='/content/drive/MyDrive/checkpoints/chest_xray', CLASS_NAMES=['NORMAL', 'PNEUMONIA'], NUM_CLASSES=1, RESIZE_SIZE=256, IMG_SIZE=224, BATCH_SIZE=32, NUM_WORKERS=0, PIN_MEMORY=False, VAL_SPLIT=0.2, HFLIP_PROB=0.5, ROTATION_DEG=10, BRIGHTNESS_JITTER=0.2, CONTRAST_JITTER=0.2, LR_PHASE1=0.001, EPOCHS_PHASE1=5, LR_PHASE2=0.0001, EPOCHS_PHASE2=10, UNFREEZE_BLOCKS=3, WEIGHT_DECAY=0.0001, GRAD_CLIP=1.0, ETA_MIN=1e-06, DROPOUT=0.4, BACKBONE_OUT_FEATURES=1536, FOCAL_GAMMA=2.0, FOCAL_ALPHA=0.25, THRESHOLD=0.5, WANDB_PROJECT='chest-xray-binary', WANDB_ENTITY='', USE_WANDB=True, USE_AMP=True, CHECKPOINT_EVERY=1, EARLY_STOP_PATIENCE=3, LOG_EVERY_N_BATCHES=20)
06:15:21 | INFO     | src.utils            | All random seeds set to 42
06:15:21 | INFO    